# Hierarchical Forecasting & Reconciliation

Companion notebook for the [Hierarchical Forecasting wiki page](https://ml-viz-ruby.vercel.app/wiki/hierarchical-forecasting).

We derive the MinT reconciliation formula from scratch, compare top-down vs bottom-up vs OLS reconciliation, and visualize coherence errors.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(5)

## 1 — Summing matrix and coherence check

In [ ]:
# Hierarchy: 1 national → 2 regions → 4 stores
# S encodes: national = sum(all), region_A = s1+s2, region_B = s3+s4, plus 4 bottom series
S = np.array([
    [1, 1, 1, 1],   # national
    [1, 1, 0, 0],   # region A
    [0, 0, 1, 1],   # region B
    [1, 0, 0, 0],   # store 1
    [0, 1, 0, 0],   # store 2
    [0, 0, 1, 0],   # store 3
    [0, 0, 0, 1],   # store 4
], dtype=float)

# True bottom-level values
b_true = np.array([10., 15., 8., 20.])
y_true = S @ b_true
print("True coherent forecasts:", y_true)

# Base forecasts (incoherent)
b_hat = b_true + rng.normal(0, 2, 4)  # add noise
y_hat = np.zeros(7)
y_hat[:3] = y_true[:3] + rng.normal(0, 3, 3)  # aggregate forecasts also noisy
y_hat[3:] = b_hat
print("Base (incoherent) forecasts:", y_hat.round(2))
print("Coherence error (national vs sum of stores):", abs(y_hat[0] - y_hat[3:].sum()).round(3))

## 2 — OLS MinT reconciliation

In [ ]:
def mint_ols(y_hat, S):
    """MinT reconciliation with W=I (OLS version)."""
    # ỹ = S (S'S)^{-1} S' y_hat = P y_hat
    P = S @ np.linalg.inv(S.T @ S) @ S.T
    return P @ y_hat

y_tilde = mint_ols(y_hat, S)
print("Reconciled forecasts (MinT-OLS):", y_tilde.round(2))
print("Coherence error after reconciliation:", abs(y_tilde[0] - y_tilde[3:].sum()).round(6))
print("Forecast error vs true:")
print("  Before:", np.abs(y_hat - y_true).mean().round(3))
print("  After: ", np.abs(y_tilde - y_true).mean().round(3))

## ✏️ Your turn — bottom-up vs top-down

In [ ]:
def bottom_up(bottom_forecasts, S):
    """Aggregate bottom-level forecasts to all levels using summing matrix."""
    # TODO(you): return y_tilde = S @ bottom_forecasts
    return ...

def top_down_proportional(top_forecast, S, historical_proportions):
    """Disaggregate top-level forecast using historical proportions."""
    # TODO(you): multiply top_forecast by each proportion, aggregate with S
    # historical_proportions: (m,) bottom-level proportions summing to 1
    return ...

# Test
hist_props = b_true / b_true.sum()
y_bu   = bottom_up(b_hat, S)
y_td   = top_down_proportional(y_hat[0], S, hist_props)

print("Bottom-up reconciled:", y_bu.round(2))
print("Top-down reconciled: ", y_td.round(2))
print("Bottom-up coherence error:", abs(y_bu[0] - y_bu[3:].sum()).round(6))
print("Top-down coherence error: ", abs(y_td[0] - y_td[3:].sum()).round(6))

<details><summary>Solution</summary>

```python
def bottom_up(bottom_forecasts, S):
    return S @ bottom_forecasts

def top_down_proportional(top_forecast, S, historical_proportions):
    bottom_level = top_forecast * historical_proportions
    return S @ bottom_level
```
</details>